In [1]:
from openbb import obb
import pandas as pd
import numpy as np
from scipy.stats import norm
import datetime as dt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import QuantLib as ql
import matplotlib.pyplot as plt
import plotly as pl
import plotly.graph_objects as go
import warnings
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from plotly.subplots import make_subplots
import requests

In [2]:
def FetchRates(Start_Date=None, End_Date=None):
    treasury_data = obb.fixedincome.government.treasury_rates(start_date=Start_Date, end_date=End_Date, provider="federal_reserve").to_df()
    fed_funds = obb.fixedincome.rate.effr(start_date=Start_Date, end_date=End_Date, provider="federal_reserve").to_df()[['rate']].rename(columns={'rate': 'FedFunds'})
    sofr_data = obb.fixedincome.rate.sofr(start_date=Start_Date, end_date=End_Date, provider="federal_reserve").to_df()[['rate']].rename(columns={'rate': 'SOFR'})

    #Merge
    rates_data = treasury_data.join([fed_funds, sofr_data], how='outer')
    #Reset Index
    rates_data = rates_data.rename_axis('Date').reset_index()

    #Renaming Columns
    rates_data.rename(columns={
        "month_1": "1Mo", "month_2": "2Mo", "month_3": "3Mo", "month_6": "6Mo", 
        "year_1": "1Yr", "year_2": "2Yr", "year_3": "3Yr", 
        "year_5": "5Yr", "year_7": "7Yr", "year_10": "10Yr", "year_30": "30Yr"
    }, inplace=True)

    # Filter and Sort
    cols = ['Date', 'FedFunds', 'SOFR', '1Mo', '3Mo', '6Mo', '1Yr', '2Yr', '3Yr', '5Yr', '7Yr', '10Yr', '30Yr']
    rates_data = rates_data[cols].sort_values('Date')

    # Removing rows where primary benchmarks are missing (weekends/holidays)
    rates_data = rates_data.dropna(subset=['FedFunds', '1Mo'])
    
    # Forward Fill SOFR (Modern syntax)
    rates_data['SOFR'] = rates_data['SOFR'].ffill()
    
    return rates_data

In [3]:
start_date = "2005-12-01"
end_date = dt.datetime.today().strftime('%Y-%m-%d')

rates_data=FetchRates(start_date, end_date)

In [4]:
# ─── Cell: PCA Residual Signal ────────────────────────────────────────────────
# Mode abstraction: mode parameter accepted by all functions.
# Currently 'EOD' only. 'intraday' and '5day' will route through IBKR TWS
# via the data abstraction layer — this function requires no changes.
# ──────────────────────────────────────────────────────────────────────────────

def compute_pca_residuals(df, window=20, n_components=2, mode='EOD'):
    """
    Compute rolling PCA residuals for all 10 treasury tenors.

    For each day i, fits StandardScaler + PCA(n_components) on the preceding
    `window` daily yield changes. The test observation is diff_df.iloc[i] —
    NOT iloc[i-1], which would include the test point inside the fitting window.
    The date appended is df['Date'].iloc[i] to stay aligned with the test obs.

    Residual = actual yield change (bps) − PCA reconstruction (bps).
    Eigenvalue scaling is omitted — it cancels in the round-trip reconstruction.

    Parameters
    ----------
    df : pd.DataFrame
        Rate data in the FetchRates() schema (Date + 10 tenor columns).
        For mode='intraday': same column schema supplied by IBKR data layer.
        For mode='5day':     same column schema supplied by 5-day snapshot layer.
    window : int
        Rolling PCA fitting window in trading days (default 20).
    n_components : int
        Number of PCs used for reconstruction (default 2 = Level + Slope).
    mode : str
        Data mode — 'EOD', 'intraday', or '5day'. Passed through to plots.

    Returns
    -------
    residuals_df : pd.DataFrame
        Daily residuals per tenor (bps), indexed by date.
    cumulative_residuals_df : pd.DataFrame
        Cumulative sum of daily residuals per tenor (bps), indexed by date.
    """
    tenors = ['1Mo', '3Mo', '6Mo', '1Yr', '2Yr', '3Yr', '5Yr', '7Yr', '10Yr', '30Yr']
    diff_df = df[tenors].diff()

    residuals_list = []
    dates = []

    for i in range(window + 1, len(df)):
        # ── Fitting window: rows [i-window, i) — excludes test observation ──
        window_data = diff_df.iloc[i - window:i].dropna()
        if len(window_data) < window:
            continue  # skip if window shrinks due to NaN rows at series start

        # ── Test observation: row i (the day we're computing the residual for) ──
        test_obs = diff_df.iloc[i].values.reshape(1, -1)
        if np.any(np.isnan(test_obs)):
            continue

        # ── Fit scaler + PCA on the training window only ──
        scaler = StandardScaler()
        scaled_window = scaler.fit_transform(window_data)

        pca = PCA(n_components=n_components)
        pca.fit(scaled_window)

        # ── Project test observation into standardized space ──
        # scaler.transform (not fit_transform) — keeps test obs in the
        # training scaler's standardized space, not a new one
        scaled_test = scaler.transform(test_obs)

        # ── Reconstruct via PC1 + PC2 projection then inversion ──
        # Eigenvalue scaling not applied — cancels in round-trip reconstruction
        scores = pca.transform(scaled_test)                     # (1, n_components)
        reconstructed_scaled = pca.inverse_transform(scores)    # (1, 10) in scaled space
        reconstructed = scaler.inverse_transform(reconstructed_scaled)  # (1, 10) in bps

        # ── Residual = actual − reconstruction ──
        residual = (test_obs.flatten() - reconstructed.flatten()) *100 # convert to bps
        residuals_list.append(residual)
        dates.append(df['Date'].iloc[i])  # aligned to test obs, not fit window end

    residuals_df = pd.DataFrame(residuals_list, columns=tenors, index=dates)
    cumulative_residuals_df = residuals_df.cumsum()

    return residuals_df, cumulative_residuals_df


def plot_daily_residuals(residuals_df, mode='EOD'):
    """Plot daily PCA residuals for all 10 tenors on a single Plotly chart."""
    tenors = residuals_df.columns.tolist()
    colors = [
        '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
        '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'
    ]

    fig = go.Figure()
    for i, tenor in enumerate(tenors):
        fig.add_trace(go.Scatter(
            x=residuals_df.index,
            y=residuals_df[tenor],
            mode='lines',
            name=tenor,
            line=dict(width=1.5, color=colors[i]),
            hovertemplate=f'<b>{tenor}:</b> %{{y:.2f}} bps<extra></extra>'
        ))

    fig.add_hline(y=0, line_dash='dash', line_color='black', line_width=1)
    fig.update_layout(
        title=dict(
            text=(
                f'<b>Daily PCA Residuals — All Tenors</b><br>'
                f'Mode: {mode} | Reconstruction via PC1 (Level) + PC2 (Slope)'
            ),
            x=0.5, font=dict(size=18)
        ),
        xaxis_title='Date',
        yaxis_title='Residual (bps)',
        template='plotly_white',
        hovermode='x unified',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
        margin=dict(l=50, r=50, t=110, b=50),
        height=600
    )
    return fig


def plot_cumulative_residuals(cumulative_residuals_df, mode='EOD'):
    """Plot cumulative PCA residuals for all 10 tenors on a single Plotly chart."""
    tenors = cumulative_residuals_df.columns.tolist()
    colors = [
        '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
        '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'
    ]

    fig = go.Figure()
    for i, tenor in enumerate(tenors):
        fig.add_trace(go.Scatter(
            x=cumulative_residuals_df.index,
            y=cumulative_residuals_df[tenor],
            mode='lines',
            name=tenor,
            line=dict(width=1.5, color=colors[i]),
            hovertemplate=f'<b>{tenor}:</b> %{{y:.2f}} bps<extra></extra>'
        ))

    fig.add_hline(y=0, line_dash='dash', line_color='black', line_width=1)
    fig.update_layout(
        title=dict(
            text=(
                f'<b>Cumulative PCA Residuals — All Tenors</b><br>'
                f'Mode: {mode} | Reconstruction via PC1 (Level) + PC2 (Slope)'
            ),
            x=0.5, font=dict(size=18)
        ),
        xaxis_title='Date',
        yaxis_title='Cumulative Residual (bps)',
        template='plotly_white',
        hovermode='x unified',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
        margin=dict(l=50, r=50, t=110, b=50),
        height=600
    )
    return fig


# ── Execute ───────────────────────────────────────────────────────────────────
pca_window = 20   # 20-day rolling fit window (distinct from rolling_window=5 above)
mode = 'EOD'

residuals_df, cumulative_residuals_df = compute_pca_residuals(
    rates_data, window=pca_window, n_components=2, mode=mode
)
residuals_df.index = pd.to_datetime(residuals_df.index)

print(f"Residuals computed: {len(residuals_df)} trading days")
print(f"Date range: {residuals_df.index[0].date()} → {residuals_df.index[-1].date()}")
print(f"\nDaily residuals tail (bps):")
print(residuals_df.tail(3).round(4).to_string())

# ── Plot daily residuals ──────────────────────────────────────────────────────
fig_daily = plot_daily_residuals(residuals_df, mode=mode)
fig_daily.show()

# ── Plot cumulative residuals ─────────────────────────────────────────────────
fig_cumulative = plot_cumulative_residuals(cumulative_residuals_df, mode=mode)
fig_cumulative.show()


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

Residuals computed: 5073 trading days
Date range: 2006-01-03 → 2026-04-13

Daily residuals tail (bps):
               1Mo     3Mo     6Mo     1Yr     2Yr     3Yr     5Yr     7Yr    10Yr    30Yr
2026-04-09  0.0036 -0.0056 -0.0147  0.0067  0.0126  0.0154  0.0063  0.0122  0.0075  0.0051
2026-04-10  0.0018  0.0068 -0.0031 -0.0101 -0.0076 -0.0133 -0.0021 -0.0062  0.0029  0.0125
2026-04-13 -0.0001  0.0149  0.0061 -0.0208 -0.0483 -0.0330 -0.0168 -0.0085  0.0120  0.0350


In [5]:
# ─── Cell: Stationarity Tests ─────────────────────────────────────────────────
# ADF:  H0 = unit root (non-stationary). Reject at p < 0.05 → stationary evidence.
# KPSS: H0 = stationary.                 Fail to reject p > 0.05 → stationary evidence.
# Joint classification:
#   ADF p < 0.05 AND KPSS p > 0.05  → Stationary    (eligible for trading)
#   ADF p > 0.05 AND KPSS p < 0.05  → Non-Stationary (monitor only)
#   Both reject or both fail to reject → Ambiguous   (investigate further)
# ──────────────────────────────────────────────────────────────────────────────

tenors = ['1Mo', '3Mo', '6Mo', '1Yr', '2Yr', '3Yr', '5Yr', '7Yr', '10Yr', '30Yr']

# ── 1. Full-sample ADF + KPSS ─────────────────────────────────────────────────
summary_rows = []

with warnings.catch_warnings():
    warnings.simplefilter('ignore')  # suppress KPSS interpolation boundary warnings

    for tenor in tenors:
        series = residuals_df[tenor].dropna()

        # ADF: null = unit root. p < 0.05 → reject null → stationary evidence
        adf_result = adfuller(series, autolag='AIC')
        adf_stat, adf_pval = adf_result[0], adf_result[1]

        # KPSS: null = stationary. p > 0.05 → fail to reject → stationary evidence
        # regression='c': level stationarity — appropriate for zero-mean residuals
        kpss_result = kpss(series, regression='c', nlags='auto')
        kpss_stat, kpss_pval = kpss_result[0], kpss_result[1]

        # Classification uses raw floats before formatting
        adf_stationary  = adf_pval  < 0.05
        kpss_stationary = kpss_pval > 0.05

        if adf_stationary and kpss_stationary:
            conclusion = 'Stationary'      # both tests agree: mean-reverting
        elif not adf_stationary and not kpss_stationary:
            conclusion = 'Non-Stationary'  # both tests agree: trending / random walk
        else:
            conclusion = 'Ambiguous'       # tests disagree: possibly trend-stationary

        summary_rows.append({
            'Tenor':      tenor,
            'ADF Stat':   round(adf_stat,  4),
            'ADF p-val':  '< 0.0001' if adf_pval < 0.0001 else f'{adf_pval:.4f}',
            'KPSS Stat':  round(kpss_stat, 4),
            'KPSS p-val': round(kpss_pval, 4),
            'Conclusion': conclusion
        })

summary_df = pd.DataFrame(summary_rows).set_index('Tenor')

print("=" * 74)
print("Full-Sample Stationarity Tests — Daily PCA Residuals")
print("ADF:  H0 = unit root.    Reject at p < 0.05  → stationary evidence")
print("KPSS: H0 = stationary.   p > 0.05 (no reject) → stationary evidence")
print("KPSS p-values bounded [0.01, 0.10] by statsmodels interpolation table")
print("=" * 74)
print(summary_df.to_string())
print("=" * 74)

stationary_tenors    = summary_df[summary_df['Conclusion'] == 'Stationary'].index.tolist()
nonstationary_tenors = summary_df[summary_df['Conclusion'] == 'Non-Stationary'].index.tolist()
ambiguous_tenors     = summary_df[summary_df['Conclusion'] == 'Ambiguous'].index.tolist()

print(f"\nStationary    (eligible for trading): {stationary_tenors}")
print(f"Non-Stationary (monitor only):        {nonstationary_tenors}")
print(f"Ambiguous      (investigate further): {ambiguous_tenors}")


# ── 2. Rolling 60-day ADF p-values ────────────────────────────────────────────
# Date alignment: p-value at index[i-1] uses the window ending on that date.
# First valid date: residuals_df.index[59] (60 observations required).
rolling_adf_window = 60
rolling_adf_pvals  = {}

with warnings.catch_warnings():
    warnings.simplefilter('ignore')

    for tenor in tenors:
        series = residuals_df[tenor].dropna()
        pvals      = []
        roll_dates = []

        for i in range(rolling_adf_window, len(series) + 1):
            window_series = series.iloc[i - rolling_adf_window:i]
            try:
                result = adfuller(window_series, autolag='AIC')
                pvals.append(result[1])
            except Exception:
                pvals.append(np.nan)
            roll_dates.append(series.index[i - 1])  # date of last obs in window

        rolling_adf_pvals[tenor] = pd.Series(pvals, index=roll_dates)

rolling_adf_df = pd.DataFrame(rolling_adf_pvals)

print(f"\nRolling ADF computed: {len(rolling_adf_df)} dates")
print(f"Date range: {rolling_adf_df.index[0].date()} → {rolling_adf_df.index[-1].date()}")


# ── 3. Plot rolling ADF p-values ──────────────────────────────────────────────
colors = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
    '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'
]

fig_adf = go.Figure()

for i, tenor in enumerate(tenors):
    fig_adf.add_trace(go.Scatter(
        x=rolling_adf_df.index,
        y=rolling_adf_df[tenor],
        mode='lines',
        name=tenor,
        line=dict(width=1.5, color=colors[i]),
        hovertemplate=f'<b>{tenor}:</b> p = %{{y:.4f}}<extra></extra>'
    ))

# Significance threshold
fig_adf.add_hline(
    y=0.05,
    line_dash='dash',
    line_color='red',
    line_width=1.5,
    annotation_text='p = 0.05',
    annotation_position='top right',
    annotation_font=dict(color='red', size=12)
)

fig_adf.update_layout(
    title=dict(
        text=(
            '<b>Rolling 60-Day ADF p-Values — All Tenors</b><br>'
            'Below dashed line (p &lt; 0.05): stationary regime  |  '
            'Above: non-stationary regime'
        ),
        x=0.5, font=dict(size=18)
    ),
    xaxis_title='Date',
    yaxis_title='ADF p-value',
    yaxis=dict(range=[0, 1.05], tickformat='.2f'),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    margin=dict(l=50, r=50, t=110, b=50),
    height=600
)

fig_adf.show()


Full-Sample Stationarity Tests — Daily PCA Residuals
ADF:  H0 = unit root.    Reject at p < 0.05  → stationary evidence
KPSS: H0 = stationary.   p > 0.05 (no reject) → stationary evidence
KPSS p-values bounded [0.01, 0.10] by statsmodels interpolation table
       ADF Stat ADF p-val  KPSS Stat  KPSS p-val  Conclusion
Tenor                                                       
1Mo    -18.1944  < 0.0001     0.0286      0.1000  Stationary
3Mo    -15.9489  < 0.0001     0.4030      0.0758  Stationary
6Mo    -14.9396  < 0.0001     0.0151      0.1000  Stationary
1Yr    -16.8036  < 0.0001     0.0955      0.1000  Stationary
2Yr    -18.2643  < 0.0001     0.0959      0.1000  Stationary
3Yr    -20.0215  < 0.0001     0.0394      0.1000  Stationary
5Yr    -15.3296  < 0.0001     0.0592      0.1000  Stationary
7Yr    -13.5527  < 0.0001     0.1523      0.1000  Stationary
10Yr   -20.2250  < 0.0001     0.0989      0.1000  Stationary
30Yr   -17.1041  < 0.0001     0.0367      0.1000  Stationary

Stationar

KeyboardInterrupt: 

In [6]:
# ─── Cell: ACF/PACF Analysis & Curve Segment Regime Filter ───────────────────
# Purpose: determine natural mean reversion horizon per tenor and whether
# the 20-day PCA window and 60-day Z-score window are appropriate.
# ──────────────────────────────────────────────────────────────────────────────

MAX_LAGS = 30
tenors   = ['1Mo', '3Mo', '6Mo', '1Yr', '2Yr', '3Yr', '5Yr', '7Yr', '10Yr', '30Yr']

# ── 1. Compute ACF + PACF ─────────────────────────────────────────────────────
acf_results  = {}
pacf_results = {}
ci_bounds    = {}

for tenor in tenors:
    series = residuals_df[tenor].dropna()
    acf_results[tenor]  = acf(series,  nlags=MAX_LAGS, fft=True)
    pacf_results[tenor] = pacf(series, nlags=MAX_LAGS, method='ywa')
    ci_bounds[tenor]    = 1.96 / np.sqrt(len(series))


# ── 2. Summary table helpers ──────────────────────────────────────────────────
def _first_acf_crossing(acf_vals, ci):
    """First lag ≥ 1 where |ACF| drops below the 95% CI bound."""
    return next(
        (lag for lag in range(1, len(acf_vals)) if abs(acf_vals[lag]) < ci),
        None
    )

def _decay_type(acf_vals, first_crossing, ci):
    """
    Sharp Cutoff — ACF crosses CI within lag 4 AND the drop is abrupt
                   (previous lag was still significant).
    Gradual Decay — ACF tapers slowly, consistent with AR persistence.
    Never Crosses  — possible unit-root or very long memory behaviour.
    """
    if first_crossing is None:
        return 'Never Crosses CI'
    if first_crossing <= 4 and (first_crossing == 1 or abs(acf_vals[first_crossing - 1]) > ci):
        return 'Sharp Cutoff'
    return 'Gradual Decay'

def _classify_acf(first_crossing):
    if first_crossing is None:
        return 'Unit Root — Do Not Trade'
    if first_crossing <= 5:
        return 'Fast (lags 1-5) — High Signal Quality'
    if first_crossing <= 10:
        return 'Moderate (lags 6-10) — Tradeable'
    return 'Slow (lag >10) — Caution'


# ── 3. Build and print summary table ─────────────────────────────────────────
summary_rows = []
for tenor in tenors:
    acf_vals  = acf_results[tenor]
    pacf_vals = pacf_results[tenor]
    ci = ci_bounds[tenor]

    first_crossing = _first_acf_crossing(acf_vals, ci)
    sig_pacf_lags  = [lag for lag in range(1, MAX_LAGS + 1) if abs(pacf_vals[lag]) > ci]

    summary_rows.append({
        'Tenor':           tenor,
        'First ACF Cross': first_crossing if first_crossing is not None else 'Never',
        'Decay Type':      _decay_type(acf_vals, first_crossing, ci),
        'Sig. PACF Lags':  sig_pacf_lags if sig_pacf_lags else ['None'],
        'Classification':  _classify_acf(first_crossing),
    })

acf_summary_df = pd.DataFrame(summary_rows).set_index('Tenor')

print("=" * 90)
print("ACF/PACF Analysis — Mean Reversion Horizon per Tenor (Daily PCA Residuals)")
print("95% CI: ±1.96/√n  |  Max lags: 30  |  ACF: FFT  |  PACF: Yule-Walker (unbiased)")
print("=" * 90)
print(acf_summary_df.to_string())
print("=" * 90)


# ── 4. ACF/PACF subplots — 10 rows × 2 cols (ACF left | PACF right) ──────────
lags = list(range(1, MAX_LAGS + 1))

subplot_titles = []
for tenor in tenors:
    subplot_titles += [f'<b>{tenor}</b> — ACF', f'<b>{tenor}</b> — PACF']

fig_acf_pacf = make_subplots(
    rows=10, cols=2,
    subplot_titles=subplot_titles,
    shared_xaxes=False,
    vertical_spacing=0.025,
    horizontal_spacing=0.10
)

for row, tenor in enumerate(tenors, start=1):
    acf_plot  = acf_results[tenor][1:]    # lags 1..30 (drop lag-0 = 1.0)
    pacf_plot = pacf_results[tenor][1:]   # lags 1..30
    ci = ci_bounds[tenor]

    # ACF bars (blue)
    fig_acf_pacf.add_trace(
        go.Bar(x=lags, y=acf_plot, marker_color='#1f77b4',
               showlegend=False,
               hovertemplate='Lag %{x}: %{y:.4f}<extra></extra>'),
        row=row, col=1
    )
    # PACF bars (orange)
    fig_acf_pacf.add_trace(
        go.Bar(x=lags, y=pacf_plot, marker_color='#ff7f0e',
               showlegend=False,
               hovertemplate='Lag %{x}: %{y:.4f}<extra></extra>'),
        row=row, col=2
    )
    # 95% CI bands — upper and lower — for both columns
    for col_idx in [1, 2]:
        for y_val in [ci, -ci]:
            fig_acf_pacf.add_trace(
                go.Scatter(
                    x=[0.5, MAX_LAGS + 0.5], y=[y_val, y_val],
                    mode='lines',
                    line=dict(color='#d62728', dash='dash', width=1),
                    showlegend=False, hoverinfo='skip'
                ),
                row=row, col=col_idx
            )

fig_acf_pacf.update_layout(
    title=dict(
        text=(
            '<b>ACF & PACF — Daily PCA Residuals by Tenor</b><br>'
            'Blue: ACF  |  Orange: PACF  |  Dashed red: 95% CI (±1.96/√n)'
        ),
        x=0.5, font=dict(size=18)
    ),
    template='plotly_white',
    height=2800,
    showlegend=False,
    margin=dict(l=50, r=50, t=120, b=50),
    bargap=0.15
)

# Consistent x-axis ticks across all subplots
fig_acf_pacf.update_xaxes(
    tickmode='linear', tick0=5, dtick=5, range=[0, MAX_LAGS + 1],
    title_text='Lag'
)

fig_acf_pacf.show()


# ── 5. Curve segment regime filter ────────────────────────────────────────────
def compute_segment_regime_filter(rolling_adf_df, mode='EOD'):
    """
    Compute a daily boolean flag per curve segment indicating whether
    that segment is in a non-stationary regime.

    Logic: if 2+ tenors in a segment have rolling ADF p-value > 0.05 on
    a given date, the whole segment is flagged True (blocked for new signals).
    This catches regime-wide dislocation versus idiosyncratic tenor moves.

    Segment definitions
    -------------------
    Short End   : 1Mo, 3Mo, 6Mo, 1Yr   (4 tenors — trigger at 2+)
    Belly Short : 2Yr, 3Yr, 5Yr        (3 tenors — trigger at 2+)
    Belly Long  : 7Yr, 10Yr            (2 tenors — trigger at 2+, i.e. both)
    Long End    : 30Yr                 (1 tenor  — 2+ threshold unreachable;
                                        30Yr is gated by its own rolling ADF)

    Parameters
    ----------
    rolling_adf_df : pd.DataFrame
        Rolling ADF p-values indexed by date. Columns must include all 10 tenors.
        Output of the 60-day rolling ADF computation in the stationarity cell.
    mode : str
        Data mode — 'EOD', 'intraday', or '5day'. Label only; no branching.

    Returns
    -------
    segment_regime_df : pd.DataFrame
        Boolean DataFrame indexed by date, four columns (one per segment).
        True  = non-stationary regime — no new signals for any tenor in segment.
        False = stationary regime     — signals eligible subject to other gates.
    """
    segments = {
        'Short End':   ['1Mo', '3Mo', '6Mo', '1Yr'],
        'Belly Short': ['2Yr', '3Yr', '5Yr'],
        'Belly Long':  ['7Yr', '10Yr'],
        'Long End':    ['30Yr'],           # single-tenor; flag never triggered via 2+ rule
    }

    regime = {}
    for seg_name, seg_tenors in segments.items():
        # Count non-stationary tenors per day (p > 0.05 means ADF fails to reject unit root)
        n_nonstationary = (rolling_adf_df[seg_tenors] > 0.05).sum(axis=1)
        regime[seg_name] = n_nonstationary >= 2

    return pd.DataFrame(regime, index=rolling_adf_df.index)


# ── Execute ────────────────────────────────────────────────────────────────────
segment_regime_df = compute_segment_regime_filter(rolling_adf_df, mode=mode)

_seg_tenors_map = {
    'Short End':   ['1Mo', '3Mo', '6Mo', '1Yr'],
    'Belly Short': ['2Yr', '3Yr', '5Yr'],
    'Belly Long':  ['7Yr', '10Yr'],
    'Long End':    ['30Yr'],
}

print("\nCurve Segment Regime Filter — Blocked-Day Summary")
print(f"Mode: {mode}  |  {segment_regime_df.index[0].date()} → {segment_regime_df.index[-1].date()}")
print(f"Rolling ADF window: {rolling_adf_window} days  |  Total dates: {len(segment_regime_df)}")
print()
print(f"{'Segment':<14}  {'Tenors':<28}  {'Blocked Days':>12}  {'% History':>10}")
print("─" * 72)
for seg_name, seg_tenors in _seg_tenors_map.items():
    blocked = int(segment_regime_df[seg_name].sum())
    pct     = blocked / len(segment_regime_df) * 100
    note    = ' *' if seg_name == 'Long End' else ''
    print(f"{seg_name:<14}  {', '.join(seg_tenors):<28}  {blocked:>12}  {pct:>9.1f}%{note}")
print("─" * 72)
print("* Long End (30Yr only): 2+ threshold unreachable — gated by per-tenor rolling ADF instead")


KeyboardInterrupt: 

In [7]:
# ============================================================================
# PART A: Treasury Auction Calendar Fetcher with Caching
# ============================================================================

_auction_cache = {}  # Session-level cache to avoid repeated API calls

def fetch_auction_calendar(mode='EOD'):
    """
    Fetch upcoming and recent treasury auction data from TreasuryDirect.
    Results are cached for the session.
    
    Returns:
        pd.DataFrame: Columns [security_type, auction_date, issue_date, maturity_date, tenor_label]
    """
    global _auction_cache
    
    if mode in _auction_cache:
        return _auction_cache[mode].copy()
    
    # Tenor mapping: Security description → closest tenor in our analysis
    tenor_mapping = {
        # Bills
        '4-Week': '1Mo',
        '13-Week': '3Mo',
        '26-Week': '6Mo',
        '52-Week': '1Yr',
        # Notes
        '2-Year': '2Yr',
        '3-Year': '3Yr',
        '5-Year': '5Yr',
        '7-Year': '7Yr',
        '10-Year': '10Yr',
        # Bonds
        '30-Year': '30Yr',
        # Fallbacks for other terms close to our buckets
        '6-Week': '1Mo',
        '8-Week': '3Mo',
        '17-Week': '6Mo',
        '20-Year': '30Yr',
    }
    
    url = "https://www.treasurydirect.gov/TA_WS/securities/announced"
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print(f"ERROR fetching auction calendar: {e}")
        return pd.DataFrame(columns=['security_type', 'auction_date', 'issue_date', 'maturity_date', 'tenor_label'])
    
    records = []
    
    # API returns a list directly
    securities = data if isinstance(data, list) else []
    
    for sec in securities:
        sec_type = sec.get('securityType', '')
        
        # Filter: only Bill, Note, Bond
        if sec_type not in ['Bill', 'Note', 'Bond']:
            continue
        
        # Extract fields directly from the security object
        # The 'issueDate' is when the security is issued (auctioned)
        auction_date_str = sec.get('issueDate', '')
        issue_date_str = sec.get('issueDate', '')
        maturity_date_str = sec.get('maturityDate', '')
        
        # Look up tenor label
        sec_term = sec.get('securityTerm', '')
        tenor_label = tenor_mapping.get(sec_term, None)
        
        if tenor_label and auction_date_str:  # Only include if we have a tenor and auction date
            records.append({
                'security_type': sec_type,
                'auction_date': auction_date_str,  # Use issueDate as auction date
                'issue_date': issue_date_str,
                'maturity_date': maturity_date_str,
                'tenor_label': tenor_label
            })
    
    df = pd.DataFrame(records)
    
    if not df.empty:
        # Convert date strings to datetime
        for col in ['auction_date', 'issue_date', 'maturity_date']:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors='coerce')
        
        # Sort by auction date (newest first for upcoming/recent)
        df = df.sort_values('auction_date', ascending=False).reset_index(drop=True)
    
    _auction_cache[mode] = df.copy()
    print(f"Auction calendar fetched: {len(df)} securities | Mode: {mode}")
    return df


# ============================================================================
# PART B: Dynamic Auction Suppression Window Function
# ============================================================================

def get_auction_suppression_flag(date, tenor, auction_calendar_df, mode='EOD'):
    """
    Determine auction suppression flag based on auction schedule and tenor.
    
    Args:
        date (pd.Timestamp or datetime): Check date
        tenor (str): e.g., '1Mo', '3Mo', '10Yr', '30Yr'
        auction_calendar_df (pd.DataFrame): From fetch_auction_calendar()
        mode (str): 'EOD' or other mode for compatibility
    
    Returns:
        tuple: (flag, next_auction_info)
        - flag: 'SUPPRESS', 'WARN', or 'CLEAR'
        - next_auction_info: {'next_auction_date': date or None, 'days_to_next': int or None}
    """
    
    if auction_calendar_df.empty or 'tenor_label' not in auction_calendar_df.columns:
        return ('CLEAR', {'next_auction_date': None, 'days_to_next': None})
    
    # Filter for this tenor
    tenor_auctions = auction_calendar_df[auction_calendar_df['tenor_label'] == tenor].copy()
    
    if tenor_auctions.empty:
        return ('CLEAR', {'next_auction_date': None, 'days_to_next': None})
    
    # Ensure date is datetime.date for comparison
    check_date = pd.Timestamp(date).date()
    tenor_auctions['auction_date'] = pd.to_datetime(tenor_auctions['auction_date']).dt.date
    
    # Find next auction (on or after check_date)
    future_auctions = tenor_auctions[tenor_auctions['auction_date'] >= check_date]
    
    if future_auctions.empty:
        # No future auctions; find most recent
        return ('CLEAR', {'next_auction_date': None, 'days_to_next': None})
    
    next_auction_date = future_auctions.iloc[0]['auction_date']
    days_delta = (next_auction_date - check_date).days
    
    # Define suppression windows by tenor group
    bill_tenors = ['1Mo', '3Mo', '6Mo', '1Yr']
    note_tenors = ['2Yr', '3Yr', '5Yr', '7Yr', '10Yr']
    bond_tenors = ['30Yr']
    
    if tenor in bill_tenors:
        suppress_window = 2  # days before auction
    elif tenor in note_tenors:
        suppress_window = 5  # days before auction
    elif tenor in bond_tenors:
        suppress_window = 7  # days before auction
    else:
        suppress_window = 2  # default
    
    # Decision logic:
    # SUPPRESS: from (suppress_window days before) to (1 day before) auction
    # WARN: on auction day or 1 day after
    # CLEAR: otherwise
    
    if days_delta < 0:
        # Past the auction; check if it's in WARN window (day of or day after)
        if days_delta == 0 or days_delta == -1:
            return ('WARN', {'next_auction_date': next_auction_date, 'days_to_next': days_delta})
        else:
            return ('CLEAR', {'next_auction_date': next_auction_date, 'days_to_next': days_delta})
    elif 0 <= days_delta <= suppress_window:
        # Within suppression window
        if days_delta == 0 or days_delta == 1:
            return ('WARN', {'next_auction_date': next_auction_date, 'days_to_next': days_delta})
        else:
            return ('SUPPRESS', {'next_auction_date': next_auction_date, 'days_to_next': days_delta})
    else:
        # Outside window
        return ('CLEAR', {'next_auction_date': next_auction_date, 'days_to_next': days_delta})


# ============================================================================
# PART C: Signal Output Format Definition & Sample Generation
# ============================================================================

def build_signal_output(
    tenor,
    date,
    z_score,
    signal_direction,
    rolling_adf_pvalue,
    segment_regime_status,
    auction_flag,
    next_auction_date,
    days_to_next_auction,
    mode='EOD'
):
    """
    Construct signal output dictionary for a given tenor on a given date.
    
    Returns:
        dict: Complete signal with all required fields
    """
    return {
        'date': date,
        'tenor': tenor,
        'z_score': z_score,
        'signal_direction': signal_direction,  # LONG, SHORT, FLAT
        'rolling_adf_pvalue': rolling_adf_pvalue,
        'segment_regime_status': segment_regime_status,
        'auction_flag': auction_flag,  # SUPPRESS, WARN, CLEAR
        'next_auction_date': next_auction_date,
        'days_to_next_auction': days_to_next_auction,
        'mode': mode
    }


# ============================================================================
# FETCH AUCTION CALENDAR & GENERATE SAMPLE OUTPUT
# ============================================================================

print("\n" + "="*80)
print("PART A: TREASURY AUCTION CALENDAR")
print("="*80)

# Fetch auction calendar
auction_calendar = fetch_auction_calendar(mode=mode)

print(f"\nAuction Calendar Summary (first 10 records):")
print(auction_calendar.head(10).to_string())

print("\n" + "="*80)
print("PART B & C: AUCTION SUPPRESSION FLAGS & SIGNAL OUTPUT")
print("="*80)

# Generate sample output for most recent date and 10Yr tenor
if not rolling_adf_df.empty and '10Yr' in rolling_adf_df.columns:
    most_recent_date = rolling_adf_df.index[-1]
    tenor_sample = '10Yr'
    
    # Extract values from existing analysis
    z_score_val = rolling_adf_df.loc[most_recent_date, tenor_sample] if tenor_sample in rolling_adf_df.columns else 0.0
    rolling_adf_pval = rolling_adf_pvals[tenor_sample].iloc[-1] if tenor_sample in rolling_adf_pvals else np.nan
    regime_status = segment_regime_df.loc[most_recent_date, 'Belly Long'] if 'Belly Long' in segment_regime_df.columns else False
    
    # Determine signal direction (simple z-score logic)
    if z_score_val > 2.0:
        signal_dir = 'LONG'
    elif z_score_val < -2.0:
        signal_dir = 'SHORT'
    else:
        signal_dir = 'FLAT'
    
    # Get auction flag
    auction_flag, auction_info = get_auction_suppression_flag(
        most_recent_date,
        tenor_sample,
        auction_calendar,
        mode=mode
    )
    
    # Build and display sample output
    sample_signal = build_signal_output(
        tenor=tenor_sample,
        date=most_recent_date,
        z_score=float(z_score_val),
        signal_direction=signal_dir,
        rolling_adf_pvalue=float(rolling_adf_pval),
        segment_regime_status='BLOCKED' if regime_status else 'ACTIVE',
        auction_flag=auction_flag,
        next_auction_date=auction_info['next_auction_date'],
        days_to_next_auction=auction_info['days_to_next'],
        mode=mode
    )
    
    print(f"\nSample Signal Output for {tenor_sample} on {most_recent_date.date()}")
    print("─" * 80)
    for key, value in sample_signal.items():
        print(f"  {key:<25}: {value}")
    print("─" * 80)
    
    # Show what SUPPRESS, WARN, CLEAR states mean for this tenor
    print(f"\nAuction Suppression Windows for {tenor_sample}:")
    print(f"  SUPPRESS window: 5 days before auction")
    print(f"  WARN window: Auction day + 1 day after")
    print(f"  CLEAR: Outside these windows")
    print(f"  Current status: {auction_flag}")
    if auction_info['next_auction_date']:
        print(f"  Next auction: {auction_info['next_auction_date']}")
        if auction_info['days_to_next'] is not None:
            print(f"  Days to next: {auction_info['days_to_next']}")
else:
    print("\nWARNING: Cannot generate sample — required data not available")


PART A: TREASURY AUCTION CALENDAR
Auction calendar fetched: 227 securities | Mode: EOD

Auction Calendar Summary (first 10 records):
  security_type auction_date issue_date maturity_date tenor_label
0          Bill   2026-04-21 2026-04-21    2026-05-19         1Mo
1          Bill   2026-04-21 2026-04-21    2026-08-18         6Mo
2          Bill   2026-04-21 2026-04-21    2026-06-16         3Mo
3          Bill   2026-04-16 2026-04-16    2026-05-28         1Mo
4          Bill   2026-04-16 2026-04-16    2026-07-16         3Mo
5          Bill   2026-04-16 2026-04-16    2026-10-15         6Mo
6          Bill   2026-04-16 2026-04-16    2027-04-15         1Yr
7          Note   2026-04-15 2026-04-15    2029-04-15         3Yr
8          Bill   2026-04-14 2026-04-14    2026-05-12         1Mo
9          Bill   2026-04-14 2026-04-14    2026-06-09         3Mo

PART B & C: AUCTION SUPPRESSION FLAGS & SIGNAL OUTPUT


NameError: name 'rolling_adf_df' is not defined

In [ ]:
# ============================================================================
# PART A: Dual-Window Z-Score Computation
# ============================================================================

z_score_window = 60
z_score_df = pd.DataFrame(index=residuals_df.index)

for tenor in ['1Mo', '3Mo', '6Mo', '1Yr', '2Yr', '3Yr', '5Yr', '7Yr', '10Yr', '30Yr']:
    series = residuals_df[tenor]
    rolling_mean = series.rolling(window=z_score_window, min_periods=z_score_window).mean()
    rolling_std = series.rolling(window=z_score_window, min_periods=z_score_window).std()
    z_score_df[tenor] = (series - rolling_mean) / rolling_std

print(f"\n{'=' * 80}")
print("PART A: ROLLING Z-SCORE COMPUTATION")
print(f"{'=' * 80}")
print(f"Z-score window: {z_score_window} days")
print(f"Entry thresholds: ±2.0 standard deviations")
print(f"Computed: {len(z_score_df)} trading days with Z-scores")
print(f"Date range: {z_score_df.index[0].date()} → {z_score_df.index[-1].date()}")
print(f"\nZ-score tail (last 3 days):")
print(z_score_df.tail(3).round(4).to_string())


# ============================================================================
# PART B: Signal Card Generator Function
# ============================================================================

def generate_signal_card(date, tenor, mode='EOD'):
    """
    Generate a complete signal card for a given tenor on a given date.
    
    Returns a dictionary with all context needed for trading decisions:
    - Core signal metrics (residual, Z-score, direction)
    - Stationarity regime (rolling ADF p-value, classification)
    - Segment context (segment name, regime, peer tenors)
    - Auction context (flag, dates, windows)
    - Mean reversion context (ACF horizon, hold period)
    - Risk overlay (PC3 variance flag)
    - Overall eligibility with blocking reasons
    
    Args:
        date (pd.Timestamp or datetime): Analysis date
        tenor (str): e.g., '10Yr'
        mode (str): 'EOD' or other
    
    Returns:
        dict: Complete signal card
    """
    
    date = pd.Timestamp(date)
    card = {
        'date': date.date(),
        'tenor': tenor,
        'mode': mode
    }
    
    # ── 1. Core signal metrics ──────────────────────────────────────────────
    if date in residuals_df.index and tenor in residuals_df.columns:
        daily_residual = float(residuals_df.loc[date, tenor])
    else:
        daily_residual = np.nan
    card['daily_residual_bps'] = daily_residual
    
    if date in z_score_df.index and tenor in z_score_df.columns:
        z_score = float(z_score_df.loc[date, tenor])
    else:
        z_score = np.nan
    card['z_score'] = z_score
    
    # Signal direction
    if np.isnan(z_score):
        signal_direction = 'FLAT'
    elif z_score < -2.0:
        signal_direction = 'LONG'
    elif z_score > 2.0:
        signal_direction = 'SHORT'
    else:
        signal_direction = 'FLAT'
    card['signal_direction'] = signal_direction
    
    # ── 2. Stationarity regime ─────────────────────────────────────────────
    if date in rolling_adf_df.index and tenor in rolling_adf_df.columns:
        rolling_adf_pval = float(rolling_adf_df.loc[date, tenor])
    else:
        rolling_adf_pval = np.nan
    card['rolling_adf_pvalue'] = rolling_adf_pval
    
    if np.isnan(rolling_adf_pval):
        tenor_stationarity = 'UNKNOWN'
    elif rolling_adf_pval < 0.05:
        tenor_stationarity = 'STATIONARY'
    else:
        tenor_stationarity = 'NON-STATIONARY'
    card['tenor_stationarity_status'] = tenor_stationarity
    
    # ── 3. Segment context ─────────────────────────────────────────────────
    segment_map = {
        '1Mo': 'short-end', '3Mo': 'short-end', '6Mo': 'short-end', '1Yr': 'short-end',
        '2Yr': 'belly-short', '3Yr': 'belly-short', '5Yr': 'belly-short',
        '7Yr': 'belly-long', '10Yr': 'belly-long',
        '30Yr': 'long-end'
    }
    
    segment_tenors_map = {
        'short-end': ['1Mo', '3Mo', '6Mo', '1Yr'],
        'belly-short': ['2Yr', '3Yr', '5Yr'],
        'belly-long': ['7Yr', '10Yr'],
        'long-end': ['30Yr']
    }
    
    segment_name = segment_map.get(tenor, 'unknown')
    card['curve_segment'] = segment_name
    
    # Segment regime status
    if date in segment_regime_df.index:
        # Map segment names to DataFrame columns
        seg_col_map = {
            'short-end': 'Short End',
            'belly-short': 'Belly Short',
            'belly-long': 'Belly Long',
            'long-end': 'Long End'
        }
        seg_col = seg_col_map.get(segment_name)
        
        if seg_col and seg_col in segment_regime_df.columns:
            segment_blocked = bool(segment_regime_df.loc[date, seg_col])
        else:
            segment_blocked = False
    else:
        segment_blocked = False
    
    # Count non-stationary tenors in segment
    segment_tenors = segment_tenors_map.get(segment_name, [])
    non_stationary_in_segment = []
    for seg_tenor in segment_tenors:
        if date in rolling_adf_df.index and seg_tenor in rolling_adf_df.columns:
            p_val = rolling_adf_df.loc[date, seg_tenor]
            if p_val > 0.05:
                non_stationary_in_segment.append(seg_tenor)
    
    card['segment_regime_status'] = 'SUPPRESSED' if segment_blocked else 'ACTIVE'
    card['non_stationary_tenors_in_segment'] = non_stationary_in_segment
    card['n_non_stationary_in_segment'] = len(non_stationary_in_segment)
    
    # ── 4. Auction context ────────────────────────────────────────────────
    auction_flag, auction_info = get_auction_suppression_flag(
        date, tenor, auction_calendar, mode=mode
    )
    card['auction_flag'] = auction_flag
    card['next_auction_date'] = auction_info['next_auction_date']
    card['days_to_next_auction'] = auction_info['days_to_next']
    
    # ── 5. Mean reversion context ──────────────────────────────────────────
    if tenor in acf_summary_df.index:
        first_cross = acf_summary_df.loc[tenor, 'First ACF Cross']
        if first_cross == 'Never':
            acf_horizon = np.nan
        else:
            acf_horizon = int(first_cross)
    else:
        acf_horizon = np.nan
    
    card['acf_mean_reversion_horizon_days'] = acf_horizon
    if not np.isnan(acf_horizon):
        card['expected_hold_period_days'] = f"{max(1, acf_horizon - 2)}-{acf_horizon + 2}"
    else:
        card['expected_hold_period_days'] = "N/A"
    
    # ── 6. PC3 variance overlay ───────────────────────────────────────────
    # Compute PC3 variance on this date (using 20-day rolling window like residuals)
    if date in residuals_df.index and len(residuals_df.loc[:date]) >= 20:
        # Fit PCA with 3 components on data through this date
        pca_window = 20
        date_idx = residuals_df.index.get_loc(date)
        
        if date_idx >= pca_window:
            window_data = residuals_df.iloc[date_idx - pca_window:date_idx]
            tenors_list = ['1Mo', '3Mo', '6Mo', '1Yr', '2Yr', '3Yr', '5Yr', '7Yr', '10Yr', '30Yr']
            
            try:
                scaler = StandardScaler()
                scaled = scaler.fit_transform(window_data[tenors_list])
                pca_3 = PCA(n_components=3)
                pca_3.fit(scaled)
                pc3_var = pca_3.explained_variance_ratio_[2] if pca_3.n_components_ >= 3 else np.nan
            except:
                pc3_var = np.nan
        else:
            pc3_var = np.nan
    else:
        pc3_var = np.nan
    
    card['pc3_explained_variance'] = float(pc3_var) if not np.isnan(pc3_var) else np.nan
    
    # Flag as ELEVATED if above 80th percentile (to be computed from full history)
    # For now, use heuristic: if > 0.15 (rough 80th percentile)
    pc3_elevated = not np.isnan(pc3_var) and pc3_var > 0.15
    card['pc3_elevated_flag'] = pc3_elevated
    
    # ── 7. Overall trade eligibility ───────────────────────────────────────
    blocked_reasons = []
    
    # Priority order: AUCTION-SUPPRESS, SEGMENT-SUPPRESSED, NON-STATIONARY, PC3-ELEVATED, FLAT-SIGNAL
    if auction_flag == 'SUPPRESS':
        blocked_reasons.append('AUCTION-SUPPRESS')
    
    if segment_blocked:
        blocked_reasons.append('SEGMENT-SUPPRESSED')
    
    if tenor_stationarity == 'NON-STATIONARY':
        blocked_reasons.append('NON-STATIONARY')
    
    if pc3_elevated:
        blocked_reasons.append('PC3-ELEVATED')
    
    if signal_direction == 'FLAT':
        blocked_reasons.append('FLAT-SIGNAL')
    
    if blocked_reasons:
        card['trade_eligibility'] = f"BLOCKED ({', '.join(blocked_reasons)})"
    else:
        card['trade_eligibility'] = 'ELIGIBLE'
    
    return card


# ============================================================================
# PART C: Signal Scan Function
# ============================================================================

def scan_signals(date, mode='EOD'):
    """
    Scan all 10 tenors on a given date and return signal cards as DataFrame.
    
    Args:
        date (pd.Timestamp or datetime): Date to scan
        mode (str): 'EOD' or other
    
    Returns:
        pd.DataFrame: Signal cards sorted by absolute Z-score descending
    """
    tenors = ['1Mo', '3Mo', '6Mo', '1Yr', '2Yr', '3Yr', '5Yr', '7Yr', '10Yr', '30Yr']
    
    cards = []
    for tenor in tenors:
        card = generate_signal_card(date, tenor, mode=mode)
        cards.append(card)
    
    df = pd.DataFrame(cards)
    
    # Sort by absolute Z-score descending
    df['abs_z_score'] = df['z_score'].abs()
    df = df.sort_values('abs_z_score', ascending=False, na_position='last').reset_index(drop=True)
    df = df.drop(columns=['abs_z_score'])
    
    return df


# ── Execute Part C: Scan most recent date ──────────────────────────────────
print(f"\n{'=' * 80}")
print("PART C: SIGNAL SCAN — MOST RECENT DATE")
print(f"{'=' * 80}")

most_recent_date = z_score_df.index[-1]
signal_scan = scan_signals(most_recent_date, mode=mode)

print(f"\nSignal Scan for {most_recent_date.date()}")
print(f"Mode: {mode}\n")

# Display in readable format
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

for idx, row in signal_scan.iterrows():
    print(f"{idx + 1}. {row['tenor']:<6} | Z={row['z_score']:>7.2f} | "
          f"Dir={row['signal_direction']:<5} | {row['trade_eligibility']}")
    print(f"   Residual: {row['daily_residual_bps']:>7.2f} bps | "
          f"ADF p={row['rolling_adf_pvalue']:>6.3f} | {row['tenor_stationarity_status']}")
    if row['auction_flag'] != 'CLEAR':
        print(f"   Auction: {row['auction_flag']} (next: {row['next_auction_date']}, "
              f"in {row['days_to_next_auction']} days)")
    if row['segment_regime_status'] == 'SUPPRESSED':
        print(f"   Segment: {row['curve_segment']} — SUPPRESSED "
              f"({row['n_non_stationary_in_segment']}/tenors non-stationary)")
    print()


# ============================================================================
# PART D: Z-Score Time Series Plot
# ============================================================================

print(f"{'=' * 80}")
print("PART D: Z-SCORE TIME SERIES — ALL TENORS")
print(f"{'=' * 80}")

tenors = ['1Mo', '3Mo', '6Mo', '1Yr', '2Yr', '3Yr', '5Yr', '7Yr', '10Yr', '30Yr']
colors = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
    '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'
]

fig_zscore = go.Figure()

for i, tenor in enumerate(tenors):
    fig_zscore.add_trace(go.Scatter(
        x=z_score_df.index,
        y=z_score_df[tenor],
        mode='lines',
        name=tenor,
        line=dict(width=1.5, color=colors[i]),
        hovertemplate=f'<b>{tenor}:</b> Z = %{{y:.2f}}<extra></extra>'
    ))

# Entry thresholds at ±2.0
fig_zscore.add_hline(y=2.0, line_dash='dash', line_color='red', line_width=1.5,
                     annotation_text='Z = +2.0 (SHORT)', annotation_position='top right',
                     annotation_font=dict(color='red', size=11))
fig_zscore.add_hline(y=-2.0, line_dash='dash', line_color='green', line_width=1.5,
                     annotation_text='Z = −2.0 (LONG)', annotation_position='bottom right',
                     annotation_font=dict(color='green', size=11))
fig_zscore.add_hline(y=0, line_dash='solid', line_color='gray', line_width=1, opacity=0.3)

fig_zscore.update_layout(
    title=dict(
        text=(
            f'<b>Rolling 60-Day Z-Scores — All Tenors</b><br>'
            f'Entry thresholds at ±2.0 | Mode: {mode}'
        ),
        x=0.5, font=dict(size=18)
    ),
    xaxis_title='Date',
    yaxis_title='Z-Score (std devs)',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    margin=dict(l=50, r=50, t=110, b=50),
    height=600
)

fig_zscore.show()

print("\nZ-score plot generated successfully")

KeyError: 'bond_status'